<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/More_Api_And_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Find More Useful APIs and Tools

Your agents can only do what their tools allow — so "where do tools come from?" is a real engineering question. The answer used to be a catalog of framework-specific plugins, one bespoke integration per service. In 2026 it has three layers, and this notebook works through each:

1. **Plain functions** — anything with an API becomes a tool in ten lines (you've done this all course).
2. **MCP — the Model Context Protocol** — one open standard that lets any agent attach any compliant tool server: write the integration zero times instead of once per framework. 📎 *The MCP lesson covered the protocol; here we consume it from an agent.*
3. **Tool marketplaces over MCP** — hosted servers like **Zapier MCP** that expose thousands of app actions (Gmail, Slack, Sheets, ~9,000 apps / 30,000+ actions) through that same standard.

Plus a look at **CrewAI**, a popular framework for splitting work across multiple tool-using agents. Pins current as of **August 2026**.

## 🧭 What You'll Learn

- Turning any API into an agent tool with `@tool` + `create_agent()` (the pattern behind every "integration")
- Consuming an **MCP server** from a LangChain agent with `langchain-mcp-adapters` — tools you didn't write, discovered at runtime
- Connecting a hosted tool marketplace (**Zapier MCP**) the same way
- A minimal **CrewAI** crew: two role-based agents collaborating on a research-and-write task
- How to evaluate a new tool source: standard protocol > per-framework plugin

## 1. Setup: Environment, Keys, and Pins

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model provider (dropdown in Colab; edit the values locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

_MODEL_FOR = {
    "gemini": "google_genai:gemini-3.7-flash",
    "openai": "openai:gpt-5.6-luna",
    "anthropic": "anthropic:claude-sonnet-5",
}
MODEL = _MODEL_FOR[PROVIDER]

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = [_KEY_FOR[PROVIDER]]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "tai-aitutor==0.0.3",
            "langchain==1.3.14",
            "langchain-google-genai==4.3.2",
            "langchain-openai==1.4.1",
            "langchain-anthropic==1.5.4",
            "langchain-mcp-adapters==0.3.2",
            "mcp==1.29.0",  # langchain-mcp-adapters 0.3.2 requires mcp<2.0.0
            "crewai==1.15.12",
            "ddgs==9.14.4",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    # Locally: pip install the same pinned packages once; keys live in a .env file.

from tai_aitutor import setup_notebook

# The standard course key loader — Colab Secrets or a local .env, same code path.
setup_notebook(required_keys=REQUIRED_KEYS)

# CrewAI routes models through LiteLLM, which looks for GEMINI_API_KEY:
if os.environ.get("GOOGLE_API_KEY") and not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | model: {MODEL}")

✅ Setup complete — local | model: google_genai:gemini-3.7-flash


## 2. Layer 1: Any API Is Ten Lines from Being a Tool

Before any marketplace, remember the baseline you already own: wrap the API call in a function, decorate it, hand it to an agent. Here is web search (the `ddgs` metasearch library) attached to a `create_agent()` — 📎 the exact pattern from the LangChain 101 and agents lessons:

In [2]:
from ddgs import DDGS
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

model = init_chat_model(MODEL)


@tool
def web_search(query: str) -> str:
    """Search the web and return the top result snippets."""
    results = DDGS().text(query, max_results=3)
    if not results:
        return "No results found."
    return "\n\n".join(f"{r['title']}\n{r['body']}" for r in results)


search_agent = create_agent(
    model,
    tools=[web_search],
    system_prompt="You are a research assistant. Search the web before answering factual questions, and say what you found.",
)

result = search_agent.invoke(
    {"messages": [{"role": "user", "content": "What did OpenAI, Anthropic, or Google announce this week? One item is enough."}]}
)
print(result["messages"][-1].content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[{'type': 'text', 'text': 'One notable recent announcement comes from **OpenAI**, which introduced the **Codex app for macOS**. \n\nThe application serves as a command center for AI-assisted software development, designed to manage multi-agent workflows, run parallel development tasks, and handle long-running coding operations directly from the desktop.', 'extras': {'signature': 'Er4HCrsHARFNMg/5gx+BHt1Q1uihEpkSwlImr5tQ7855FkvMgSE+tR4Rg5gsQs7U6Nw0y/+tNlEiaYcEc/MJ66mUbamPOm3U80xhpD7zO6J8sJGGFooLHJHU7teQz+ecDQ4CVYm2pbussJgMDqEs8W8l6eEm2vk2wcoJQiZdFpr/oxdww/u+7yxDaFoV56GjMIIzbNyC6odV92o4dWOHzYmNVQASGD8y4TYX2QPxDpG6PGJaAs+qSbNmu8bl2euA2FDQn9U/nEntHevco2w+Ma155849tYBXDfnDh7c+dpKbXpiMIoGrg91+wjTj6TK/kLTKpFT/7w/KTJP0C3BtCaFyC6+0j6ohaH868hrNaSR7qLwkIF5x9ExuEKUJfL95PX6/AIo3SGxdLbSxqjSIMMsG9UDd4yN0svwDSz8hkrg9CFpY/Jrk/MOeQ9T8yEFf4ojfMUm9HVD8K6lulYPypcZN/Z9DVu/BXJo/KqpSVYz6hzsXzLMhPVyqRBOjVqkIR1HRjlOm7M/LtDqrI3qhdEXZLSBbk226H7DgpNNUy6NoE6xTNThazyoLOHUelVE/VRSg/bTyizPV7GpbochYuo2RLXWdLyQw6ecfjNYe4

**What just happened?** Ten lines turned a search library into an agent capability. This scales to any API — weather, GitHub, your company's internal services — but it scales *linearly*: every service means another hand-written wrapper, auth handling, and schema. That per-integration tax is exactly what the next layer removes.

*(A note on ecosystem churn, because it is a lesson in itself: this section used to use `DuckDuckGoSearchRun` from `langchain-community` — a package that has since been sunset. The plain-function pattern above has no such dependency to lose. For production-grade hosted search, the course's web-search lesson 📎 covers dedicated APIs like Tavily.)*

## 3. Layer 2: MCP — Tools as a Protocol, Not a Plugin

📎 *The MCP lesson built servers and clients from scratch; recap in one line: an MCP server **advertises** typed tools over a standard protocol, and any MCP-capable client — Claude, Cursor, your agent — can discover and call them.* The consequence for this lesson: a tool integration written **once, for no framework in particular**, works everywhere.

To see it end to end, write a tiny MCP server (two tools), then let a LangChain agent discover and use it via `langchain-mcp-adapters` — note the agent code never imports the server:

In [3]:
%%writefile course_stats_server.py
"""A minimal MCP server exposing two course-statistics tools (stdio transport)."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("course-stats")

# A stand-in for a real data source (a DB, an internal API, ...):
SECTION_LESSON_COUNTS = {"section 9": 7, "section 10": 8, "section 11": 7, "section 12": 9}


@mcp.tool()
def lessons_in_section(section: str) -> int:
    """Return how many lessons a course section has, e.g. 'section 12'."""
    return SECTION_LESSON_COUNTS.get(section.strip().lower(), 0)


@mcp.tool()
def total_lessons() -> int:
    """Return the total number of lessons across sections 9-12."""
    return sum(SECTION_LESSON_COUNTS.values())


if __name__ == "__main__":
    mcp.run()  # stdio transport by default

Overwriting course_stats_server.py


In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "course_stats": {
            "command": sys.executable,
            "args": ["course_stats_server.py"],
            "transport": "stdio",
        },
    }
)

mcp_tools = await mcp_client.get_tools()  # discovery: the SERVER tells US what it offers

for t in mcp_tools:
    print(f"🔧 {t.name}: {t.description}")

🔧 lessons_in_section: Return how many lessons a course section has, e.g. 'section 12'.
🔧 total_lessons: Return the total number of lessons across sections 9-12.


In [5]:
stats_agent = create_agent(
    model,
    tools=mcp_tools,  # MCP tools drop straight into the same agent constructor
    system_prompt="Answer questions about the course using the available tools.",
)

result = await stats_agent.ainvoke(
    {"messages": [{"role": "user", "content": "How many lessons are in section 12, and how many across sections 9-12?"}]}
)
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'There are **9 lessons** in section 12, and a total of **31 lessons** across sections 9–12.', 'extras': {'signature': 'EpsCCpgCARFNMg/RJv1hafSTj9BfYQ/snvHMdCGdpMcy7v/aWeClqrh+uH2LdMg06wS3Zj7d7jpm0SmwA1ARGkN1rdMLPQMMQ2OE06gESmlCF0jJr2CcbMkbSaMMXv7fzGOR8g3DMp7waFJTGDooTCyvMI/pYRgfa2ty3Qy5WIYnImfyVW7BzIAw6Any70NqnOaSdbz45Cb9rx5dynhv5uBumHNk4hnK8zKCl1pmaWGGaodxRgpZriVObIJO4krkWMqmd9VROVz2q4r53GHVRwp4EbxV61jaC9Jns0+PKe6aIUbdShpkbtr5FQ9RvHU+IyLCb5nIS/Ee4l2mjAYkmKcY/lDDv6a6B0D9x4EDrftQw+pdngF/4QR+EfiIag=='}}]


**What just happened?** The agent never imported `course_stats_server.py` — it *discovered* the tools over the protocol (names, descriptions, schemas all came from the server), then called them like any other tool. Swap our toy server for GitHub's MCP server, a database server, or a colleague's, and this cell doesn't change. That inversion — servers describe themselves; clients just connect — is why tool *catalogs* are becoming tool *servers*. (MCP tools are async; hence `ainvoke`. And the trust rule from the agents lesson doubles here: an MCP server is third-party code feeding your agent — connect to servers you trust.)

## 4. Layer 3: Tool Marketplaces — Zapier MCP

The marketplace layer is the same protocol at commercial scale. **Zapier MCP** (zapier.com/mcp) wraps Zapier's catalog — about **9,000 apps and 30,000+ actions** (Gmail, Slack, Sheets, HubSpot, …) — as a hosted MCP server. You pick which actions your personal server exposes and authenticate the apps once, on Zapier's side; your agent connects with a URL. *(Zapier retired its earlier "AI Actions" integration at actions.zapier.com in favor of exactly this.)*

In [6]:
ZAPIER_MCP_URL = ""  # @param {type:"string"}  (create yours at zapier.com/mcp — treat it like a password)

if ZAPIER_MCP_URL:
    zapier_client = MultiServerMCPClient(
        {"zapier": {"url": ZAPIER_MCP_URL, "transport": "streamable_http"}}
    )
    zapier_tools = await zapier_client.get_tools()

    print(f"{len(zapier_tools)} Zapier actions available to this agent:")
    for t in zapier_tools[:10]:
        print(f"🔧 {t.name}")

    # Attach them exactly like any other tools:
    # zapier_agent = create_agent(model, tools=zapier_tools)
    # await zapier_agent.ainvoke({"messages": [{"role": "user", "content": "Draft an email to ..."}]})
else:
    print("⏭️ No Zapier MCP URL set — create one at zapier.com/mcp and paste it above to try this section.")

⏭️ No Zapier MCP URL set — create one at zapier.com/mcp and paste it above to try this section.


**What just happened?** (If you connected:) the same discovery flow as our toy server, but the "server" is Zapier's cloud and the tools are real app actions under *your* accounts. Two production warnings before you wire this into anything: the URL embeds your credentials — **treat it as a secret** — and marketplace tools can have real-world side effects (sending mail, editing sheets), which is precisely where the human-approval patterns from the LangGraph lesson 📎 belong.

## 5. CrewAI: Multiple Agents, One Task

Tool *sources* solved, a different scaling question remains: some jobs want **several specialized agents** rather than one generalist. **CrewAI** is a popular framework for that: agents get a *role*, a *goal*, and tools; *tasks* get assigned; a *crew* runs them in sequence, passing work along. Here is a minimal researcher → writer pipeline reusing our search function:

In [7]:
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool as crew_tool

crew_llm = LLM(model={
    "gemini": "gemini/gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "anthropic/claude-sonnet-5",
}[PROVIDER])


@crew_tool("Web Search")
def crew_search(query: str) -> str:
    """Search the web and return the top result snippets."""
    return web_search.func(query)  # reuse the same underlying function from Section 2


researcher = Agent(
    role="Researcher",
    goal="Find the most significant recent development in agent communication protocols such as MCP",
    backstory="You are a meticulous AI research analyst who always verifies claims with a search.",
    tools=[crew_search],
    llm=crew_llm,
    verbose=True,
)

writer = Agent(
    role="Technical Writer",
    goal="Turn research notes into a clear, accurate summary for course students",
    backstory="You write concise technical summaries with no hype.",
    llm=crew_llm,
    verbose=True,
)

research_task = Task(
    description=(
        "Research one significant recent development in how agents talk to tools and to each other "
        "(MCP, A2A, or similar protocols). Collect 3-4 verified facts."
    ),
    expected_output="A short bullet list of verified facts with context.",
    agent=researcher,
)

writing_task = Task(
    description="Write a one-paragraph summary of the research for students finishing a course section on MCP and agent tooling.",
    expected_output="One clear paragraph, under 120 words.",
    agent=writer,
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
)


crew_result = await crew.kickoff_async()
print(crew_result)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Research one significant recent development in how agents talk to tools and to each other (MCP, A2A, or  │
│  similar protocols). Collect 3-4 verified facts.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: Introducing the Model Context Protocol \ Anthropic
The Model Context Protocol (MCP) is an open standard for connecting AI assistants to the systems where data lives, including content repositories, bu...
Tool web_search executed with result: MCP Architecture Explained: Tools, Resources & JSON-RPC ...
1 day ago · An in-depth engineering guide to the Model Context Protocol (MCP). Learn how JSON-RPC client-server primitives, transport stream...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Significant Development: Anthropic's Model Context Protocol (MCP)                                          │
│                                                                                                                 │
│  * **Launch & Standard Purpose:**                                                                               │
│    On November 25, 2024, Anthropic open-sourced the **Model Context Protocol (MCP)**, an open standard          │
│  designed to eliminate fragmented, point-to-point tool integrations. It provides a standardized, bidirectional  │
│  interface for AI agents and LLMs to discover, access, and interact with external data repositories,            │
│  development environments, and business tools.                                                                  │
│                                                                                                                 │
│  * **Underlying Protocol & Transport Architecture:**                                                            │
│    MCP is built on the **JSON-RPC 2.0** message specification and operates on a client-host-server topology.    │
│  The protocol specifies two official transport mechanisms for message exchange:                                 │
│    * **Standard Input/Output (`stdio`):** For secure, fast local process communication on the host machine.     │
│    * **Server-Sent Events (SSE) over HTTP:** For communicating with remote tools, APIs, and microservices.      │
│                                                                                                                 │
│  * **Core Functional Primitives (Resources, Tools, and Prompts):**                                              │
│    MCP structures agent-tool interactions through three standardized server-side primitives:                    │
│    * **Resources:** Read-only data endpoints providing context without side effects (e.g., file system          │
│  contents, database tables, or API documentation).                                                              │
│    * **Tools:** Executable functions with predefined schemas that models can invoke to perform external         │
│  actions (e.g., running shell commands, querying APIs, or modifying data).                                      │
│    * **Prompts:** Reusable, parameterized template workflows exposed by servers to guide models through         │
│  specific specialized tasks.                                                                                    │
│                                                                                                                 │
│  * **Broad Ecosystem and Client Adoption:**                                                                     │
│    Anthropic released open-source SDKs in TypeScript, Python, and Kotlin alongside pre-built reference servers  │
│  (including PostgreSQL, GitHub, Slack, SQLite, and Google Drive). MCP has rapidly become the de facto tool      │
│  communication standard across modern developer environments, with native client support integrated into        │
│  Claude Desktop, Cursor IDE, Zed, Sourcegraph Cody, and Replit.                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Task: Write a one-paragraph summary of the research for students finishing a course section on MCP and agent   │
│  tooling.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The Model Context Protocol (MCP), open-sourced by Anthropic, is a standardized, bidirectional interface that   │
│  connects AI agents to external tools, environments, and data sources. Built on JSON-RPC 2.0 using a            │
│  client-host-server architecture, MCP supports local process communication via standard I/O (`stdio`) and       │
│  remote access through Server-Sent Events (SSE) over HTTP. Interactions are structured around three core        │
│  server primitives: read-only Resources for data context, executable Tools for performing external actions,     │
│  and parameterized Prompts for reusable task workflows. Supported by SDKs in Python, TypeScript, and Kotlin,    │
│  MCP provides a unified standard for agentic tooling natively integrated into environments such as Claude       │
│  Desktop, Cursor, and Zed.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The Model Context Protocol (MCP), open-sourced by Anthropic, is a standardized, bidirectional interface that connects AI agents to external tools, environments, and data sources. Built on JSON-RPC 2.0 using a client-host-server architecture, MCP supports local process communication via standard I/O (`stdio`) and remote access through Server-Sent Events (SSE) over HTTP. Interactions are structured around three core server primitives: read-only Resources for data context, executable Tools for performing external actions, and parameterized Prompts for reusable task workflows. Supported by SDKs in Python, TypeScript, and Kotlin, MCP provides a unified standard for agentic tooling natively integrated into environments such as Claude Desktop, Cursor, and Zed.




## 🔑 Key Takeaways

- **Three layers of tool sourcing**: hand-wrapped functions (full control, per-service cost), MCP servers (write-once, discover-at-runtime, framework-agnostic), and MCP marketplaces like Zapier (thousands of actions, zero integration code).
- **MCP inverts the integration**: servers describe their tools; clients discover them. `langchain-mcp-adapters` turns any MCP server into `create_agent()` tools in three lines — and the same server also works from Claude, Cursor, or a hand-rolled client.
- **Marketplace power = marketplace risk**: a Zapier MCP URL is a credential, and app actions have side effects — gate them with the human-in-the-loop patterns from the LangGraph lesson.
- **CrewAI** organizes multiple role-based agents around a task pipeline; use it when subtasks need different specialists, not because more agents sounds better.
- Ecosystems churn (Zapier AI Actions retired; `langchain-community` sunset; LlamaHub's catalog model gave way to protocols) — **bet on standards over plugins**, and pin what you depend on.